# Persistence landscape

Exploratory visualization of the measured-input persistence table. Electron and proton values are experimental lower bounds. These plots do not establish a new physical law or a discovery claim.

In [ ]:
from pathlib import Path
import csv
import statistics
import matplotlib.pyplot as plt

root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
with (root / 'data' / 'persistence_table.csv').open(newline='', encoding='utf-8') as handle:
    rows = list(csv.DictReader(handle))

for row in rows:
    for key in ('Mass_MeV', 'Lifetime_s', 'Frequency_Hz', 'N', 'log10_N'):
        row[key] = float(row[key])
ordered = sorted(rows, key=lambda row: row['log10_N'])

## A. Sorted log10(N)

In [ ]:
plt.figure(figsize=(9, 4))
plt.bar([row['Particle'] for row in ordered], [row['log10_N'] for row in ordered])
plt.ylabel('log10(N)')
plt.xticks(rotation=45)
plt.title('Persistence index sorted ascending')
plt.tight_layout()
plt.show()

## B. Frequency vs lifetime

In [ ]:
plt.figure(figsize=(8, 5))
for row in rows:
    plt.scatter(row['Frequency_Hz'], row['Lifetime_s'])
    plt.annotate(row['Particle'], (row['Frequency_Hz'], row['Lifetime_s']))
plt.xscale('log')
plt.yscale('log')
plt.xlabel('frequency (Hz)')
plt.ylabel('lifetime or lower bound (s)')
plt.title('Frequency vs lifetime')
plt.tight_layout()
plt.show()

## C. Frequency vs persistence index

In [ ]:
plt.figure(figsize=(8, 5))
for row in rows:
    plt.scatter(row['Frequency_Hz'], row['N'])
    plt.annotate(row['Particle'], (row['Frequency_Hz'], row['N']))
plt.xscale('log')
plt.yscale('log')
plt.xlabel('frequency (Hz)')
plt.ylabel('N = f * tau')
plt.title('Frequency vs persistence index')
plt.tight_layout()
plt.show()

## Stability-islands exploration

This is descriptive only. The clustering rule splits the sorted sample when a gap exceeds the mean plus one population standard deviation.

In [ ]:
gaps = []
for lower, upper in zip(ordered, ordered[1:]):
    gaps.append({'lower': lower['Particle'], 'upper': upper['Particle'], 'delta': upper['log10_N'] - lower['log10_N']})
ranked_gaps = sorted(gaps, key=lambda gap: gap['delta'], reverse=True)
ranked_gaps

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist([gap['delta'] for gap in gaps], bins=6)
plt.xlabel('adjacent Delta log10(N)')
plt.ylabel('count')
plt.title('Adjacent persistence-gap histogram')
plt.tight_layout()
plt.show()

In [ ]:
threshold = statistics.mean(gap['delta'] for gap in gaps) + statistics.pstdev(gap['delta'] for gap in gaps)
cluster = 1
clusters = []
for index, row in enumerate(ordered):
    if index and gaps[index - 1]['delta'] > threshold:
        cluster += 1
    clusters.append((row['Particle'], row['log10_N'], cluster))
print('split threshold:', threshold)
clusters

In [ ]:
plt.figure(figsize=(9, 3))
for particle, log10_n, cluster in clusters:
    plt.scatter(log10_n, cluster)
    plt.annotate(particle, (log10_n, cluster))
plt.xlabel('log10(N)')
plt.ylabel('exploratory cluster')
plt.title('Adjacent-gap clustering visualization')
plt.tight_layout()
plt.show()